# KAT-VAD v2 — **second look: A3-ign on the opened sets** (Amendment 12, §23 S1–S8)

Pre-registration: `core/docs/v2/PREREG_ADDENDUM.md` **§23** (on top of §22 = E4-0, §19 = the Final, §17 = O1′).
§19 opened DoTA-CAP-eval and T2-test once for A0–A3. E4-0 then adopted **A3-ign** on DoTA-CAP-dev
(`core/docs/v2/RESULTS_E4_0.md`). This notebook reads A3-ign **once** on those already-opened sets and pairs it with
the Final's A3. **No training. Nothing re-decides (S2)**: `B` = A3-ign stands whatever the numbers show.

| step | what | GPU? |
|---|---|---|
| 0 | setup; §23 must be signed in the upload | — |
| 1 | preflight: Final outputs, E4-0's 5 A3-ign runs (J10, `ignore`), E3's A0 runs, the Final's A3 CAP-eval bake (same T2 stats); **`SECOND_LOOK_OPENED.json`** | no |
| 2 | protocol B `--final`: A3-ign × 5 on `dota_cap_eval` (560) | yes |
| 3 | T2-test (1,106 windows): `v2_guard_test` per seed, A0 + A3-ign | yes |
| 4 | `second_look_readout` (S7 gates first) → stop | no |

**Once means once (S8).** Every step skips an output already on Drive; nothing is re-scored with other settings.
Outputs: `Thesis-V2/outputs/v2_e4_0/second_look/`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read: data/, cache/ (CLIP)
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read: kat-vad/, cache/v2/, outputs/; write: outputs/v2_e4_0/second_look
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert (REPO / 'core/tools/second_look_readout.py').is_file(), 'this upload predates §23 -- re-upload branch v2'
addendum = (REPO / 'core/docs/v2/PREREG_ADDENDUM.md').read_text(encoding='utf-8')
assert '## 23. Amendment 12' in addendum, 'upload has no addendum §23'
s23 = addendum.split('## 23. Amendment 12', 1)[1]
assert 'Authorized by the user on' in s23 and 'Not in force until' not in s23, '§23 is not signed in this upload'

DATASET  = constants.DADA_ORIGIN_DATASET               # 'DADA2000_orig'
SEEDS    = list(constants.V2_E3_SEEDS)                 # 2024-2028
ARM      = constants.V2_E4_0_ARM                       # 'A3ign'
ANCHOR   = constants.V2_E4_0_ANCHOR_MODE               # 'ignore'
STEP     = 1740
CAP_EV   = constants.V2_SPLIT_DOTA_CAP_EVAL
CRN      = 'R2'

T2       = constants.DATA_ROOT / DATASET               # PARENT T2 dir: its evaluation file is T2-test
CLIP_DIR = constants.CLIP_CACHE_DIR / DATASET          # A0's T2 input
DOTA     = constants.DATA_ROOT / 'DoTA'
LABELS   = DOTA / 'labels_s8'
DOTA_S1  = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
V2_CACHE = Path(DRIVE_V2) / 'cache' / 'v2'
BAKED_T2 = V2_CACHE / f'A3_{CRN}_S' / DATASET                     # A3's T2 cache = A3-ign's (same arm input)
CAP_BAKE = V2_CACHE / f'A3_{CRN}_S' / 'DoTA_CAP_eval_s3_from_s1'  # the Final's A3 CAP-eval bake (S1)
E3       = constants.OUTPUT_ROOT / 'v2_e3'
E40      = constants.OUTPUT_ROOT / 'v2_e4_0'
FINAL    = constants.OUTPUT_ROOT / 'v2_final'
SL       = E40 / constants.V2_SL_DIR
MARKER   = SL / 'SECOND_LOOK_OPENED.json'
LOCAL = Path('/content/second_look')
LOCAL.mkdir(parents=True, exist_ok=True)
for name in ('E3', 'E40', 'FINAL', 'SL', 'CAP_BAKE'):
    print(f'{name:8s}', globals()[name])

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu
python -c "import transformers, av, einops, faiss; print('transformers', transformers.__version__)"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- scoring will be slow'
df -h /content | tail -1

In [ ]:
import json
import pickle
import shutil
import subprocess
import time

import torch
import yaml

ENV = dict(os.environ)


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (or only ``ids`` + its manifests) / file to VM disk once (bit-identical; C2).'''
    if dst.exists() and ids is None:
        return dst
    t0 = time.time()
    if src.is_dir() and ids is None:
        shutil.copytree(src, dst)
    elif src.is_dir():
        dst.mkdir(parents=True, exist_ok=True)
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for m in src.glob('*.json'):
            shutil.copy2(m, dst / m.name)
    else:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''Copy a VM-local result to Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (lesson (bi)).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path} ({path.stat().st_size / 2**20:.0f} MiB): {exc}')
        return -1


def metrics_last(path):
    lines = [l for l in path.read_text().splitlines() if l.strip()] if path.exists() else []
    return json.loads(lines[-1]) if lines else None

## 1. Preflight — every input exists and is the one §23 names; then the marker (S8)

In [ ]:
from core.data.v2_splits import load_split
from core.tools import e3_readout, final_readout, position_prior, protocol_b_eval, second_look_readout, v2_guard_test
from core.tools.v2_diagnostics import DIAG_JSON

assert (FINAL / 'OPENED.json').exists(), 'the Final never opened the sets -- this step only re-reads opened ones'
for path in (e3_readout.pb_dir(FINAL, 'A3', CAP_EV) / protocol_b_eval.CLIP_SCORES_NPZ,
             e3_readout.pb_dir(FINAL, 'A0', CAP_EV) / protocol_b_eval.CLIP_SCORES_NPZ,
             FINAL / e3_readout.REPORTS / e3_readout.PRIOR_DIR / CAP_EV / position_prior.PRIOR_NPZ,
             e3_readout.pb_dir(E40, ARM, constants.V2_SPLIT_DOTA_CAP_DEV) / protocol_b_eval.READOUT_JSON,
             *[FINAL / final_readout.T2_TEST_DIR / f's{s}' / v2_guard_test.GUARD_JSON for s in SEEDS]):
    assert path.exists(), f'missing {path} (the Final / E4-0 must be on Drive)'


def run_dir(arm, seed):
    return (E40 if arm == ARM else E3) / arm / f's{seed}'


def ckpt(arm, seed):
    return run_dir(arm, seed) / 'checkpoint_last.pt'


for seed in SEEDS:                       # J10 on every checkpoint read here, and the diag O1' reads beside it
    for arm in ('A0', ARM):
        step, last = ckpt_step(ckpt(arm, seed)), metrics_last(run_dir(arm, seed) / 'metrics.jsonl')
        assert last is not None and step == last['global_step'] == STEP, f'{arm} s{seed}: J10 (ckpt {step})'
        assert (run_dir(arm, seed) / e3_readout.DIAG_DIR / DIAG_JSON).exists(), f'{arm} s{seed}: no diag'
    cfg = yaml.safe_load((run_dir(ARM, seed) / 'config.yaml').read_text())
    assert cfg['loss']['dvs_anchor_mode'] == ANCHOR, f'{ARM} s{seed} was not trained with {ANCHOR}'

cap_ev = load_split(CAP_EV, final=True)          # §23 S1: re-reads the set §19 opened
assert len(cap_ev) == constants.V2_SL_CAP_EVAL_CLIPS, len(cap_ev)
bake = json.loads((CAP_BAKE / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
t2b = json.loads((BAKED_T2 / constants.V2_INPUT_MANIFEST_FILENAME).read_text())
assert bake['train_ids_sha1'] == t2b['train_ids_sha1'], 'CAP-eval bake was fit on another T2 split'
assert (bake['crn'], bake['motion']) == (cfg['v2']['crn'], cfg['v2']['motion']), (bake, cfg['v2'])
print(f'inputs OK: 10 checkpoints at step {STEP}; CAP-eval bake = A3-ign arm, T2 stats {t2b["train_ids_sha1"][:12]}')

if not MARKER.exists():
    SL.mkdir(parents=True, exist_ok=True)
    MARKER.write_text(json.dumps({
        'opened_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        'addendum': '§23 Amendment 12', 'arm': ARM, 'label': second_look_readout.LABEL,
        'sets': [CAP_EV, 't2_test'],
    }, indent=2))
    print('second look OPENED now:', MARKER)
else:
    print('resuming -- second look opened on', json.loads(MARKER.read_text())['opened_utc'])

L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'DoTA_s1_ncc_cap_eval', cap_ev)
L_CAP = stage(CAP_BAKE, LOCAL / 'stage' / 'A3_CAP_eval')
L_DOTA_LABELS = stage(LABELS / constants.DEFS_FILENAME,
                      LOCAL / 'stage' / 'dota_labels' / constants.DEFS_FILENAME).parent

## 2. Protocol B `--final` — A3-ign × 5 on DoTA-CAP-eval (S1)

In [ ]:
out = e3_readout.pb_dir(SL, ARM, CAP_EV)
if (out / protocol_b_eval.CLIP_SCORES_NPZ).exists():
    print(f'{ARM}/{CAP_EV}: already scored -- not re-scored (S8)')
else:
    local = LOCAL / 'pb'
    runs = [x for s in SEEDS for x in ('--run', f's{s}', ckpt(ARM, s))]
    run([sys.executable, '-m', 'core.tools.protocol_b_eval', *runs, '--final',
         '--input-dir', L_CAP, '--s1-dir', L_DOTA_S1,
         '--metadata', DOTA / 'metadata_val.json', '--split-file', DOTA / 'val_split.txt',
         '--data-dir', L_DOTA_LABELS, '--split', CAP_EV, '--out-dir', local, '--device', 'auto'],
        log=LOCAL / f'{ARM}_{CAP_EV}_pb.log')
    publish(local, out)
r = json.loads((out / protocol_b_eval.READOUT_JSON).read_text())
assert r['clips'] == constants.V2_SL_CAP_EVAL_CLIPS and set(r['global_steps'].values()) == {STEP}, r['global_steps']
print(f'{ARM} scored on {r["clips"]} DoTA-CAP-eval clips at step {STEP}')

## 3. T2-test, per seed — `v2_guard_test` (A0 + A3-ign; O1′ vs A0 of the same seed, S6)

In [ ]:
L_T2 = stage(T2, LOCAL / 'stage' / 'corpus')
test = json.loads((L_T2 / constants.FRAME_LABELS_TEST_FILENAME).read_text())
assert len(test) == constants.V2_SL_T2_TEST_WINDOWS, f'{len(test)} T2-test windows'
L_IN = {'A0': stage(CLIP_DIR, LOCAL / 'stage' / 'A0_T2'), ARM: stage(BAKED_T2, LOCAL / 'stage' / 'A3_T2')}
for seed in SEEDS:
    out = SL / final_readout.T2_TEST_DIR / f's{seed}'
    if (out / v2_guard_test.GUARD_JSON).exists():
        print(f's{seed}: already scored -- not re-scored (S8)')
        continue
    local = LOCAL / 't2_test' / f's{seed}'
    cmd = [sys.executable, '-m', 'core.tools.v2_guard_test', '--data-dir', L_T2, '--a0', 'A0',
           '--out-dir', local, '--device', 'auto']
    for arm in ('A0', ARM):
        cmd += ['--arm', arm, ckpt(arm, seed), L_IN[arm],
                '--val-diag', arm, run_dir(arm, seed) / e3_readout.DIAG_DIR / DIAG_JSON]
    run(cmd, log=LOCAL / f't2_test_s{seed}.log')
    publish(local, out)
    r = json.loads((out / v2_guard_test.GUARD_JSON).read_text())
    assert all(r['runs'][a]['windows'] == constants.V2_SL_T2_TEST_WINDOWS for a in ('A0', ARM)), f's{seed}'
print('T2-test scored for', len(SEEDS), 'seeds')

## 4. Read-out (S7 gates, then S3–S6) — no GPU

In [ ]:
run([sys.executable, '-m', 'core.tools.second_look_readout', '--e3-dir', E3, '--e4-dir', E40,
     '--final-dir', FINAL, '--sl-dir', SL])
print((SL / e3_readout.REPORTS / second_look_readout.READOUT_MD).read_text())

## 5. Record — then **stop**

Paste `v2_e4_0/second_look/REPORTS/second_look_readout.md` back to Claude (and the `.json` if asked). The sentence is
fixed by §23 S5; `B` = A3-ign stands whatever it says (S2). Nothing is re-run, re-scored or re-tuned after reading it.
`t2_test/s*/o1prime_t2test.md` also prints the pilot's Q5 wording — ignore it here; the T2-test table is the read-out's.